In [ ]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("ReadParquetWriteHive") \
    .config("spark.driver.memory", "6g")\
    .enableHiveSupport() \
    .getOrCreate()

# 2. 读取 Parquet 文件
parquet_file_path = "/home/lst/my-spark/star_schema/fact_order_items"
df = spark.read.parquet(parquet_file_path)
df.show(5)
df.show()

26/04/18 10:07:50 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/18 10:07:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 10:07:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+------------+-------------------+----+-----+-------+-----------+-----------+-----------+---------------+--------------+--------------+--------------------+-----------+-----------------+
|item_id|increment_id|         created_at|year|month|  price|qty_ordered|total_value|grand_total|discount_amount|        status|payment_method|                 sku|Customer ID|  category_name_1|
+-------+------------+-------------------+----+-----+-------+-----------+-----------+-----------+---------------+--------------+--------------+--------------------+-----------+-----------------+
| 211140|   100147451|2016-07-01 00:00:00|2016|    7|96499.0|          1|    96499.0|      96499|              0|      canceled| ublcreditcard|Apple iPhone 6S 64GB|          8|Mobiles & Tablets|
| 211211|   100147493|2016-07-01 00:00:00|2016|    7|  805.0|          1|      805.0|        805|              0|order_refunded|           cod|    noritake_NTM163M|         41|    Home & Living|
| 211236|   100147514|201

In [2]:
df.printSchema()

root
 |-- item_id: string (nullable = true)
 |-- increment_id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- qty_ordered: string (nullable = true)
 |-- total_value: double (nullable = true)
 |-- grand_total: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- sku: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- category_name_1: string (nullable = true)



In [3]:

spark.sql("CREATE DATABASE IF NOT EXISTS eco_data COMMENT 'E-commerce data warehouse'")

26/04/18 10:13:12 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/18 10:13:12 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/04/18 10:13:13 WARN ObjectStore: Failed to get database eco_data, returning NoSuchObjectException
26/04/18 10:13:13 WARN ObjectStore: Failed to get database eco_data, returning NoSuchObjectException
26/04/18 10:13:13 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
26/04/18 10:13:13 WARN ObjectStore: Failed to get database eco_data, returning NoSuchObjectException


DataFrame[]

In [5]:
spark.sql("show databases").show()

+--------------+
|     namespace|
+--------------+
|       default|
|      eco_data|
|       test_db|
|wsl_local_test|
+--------------+



In [6]:
spark.sql("use eco_data")

DataFrame[]

In [ ]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.fact_order_items (
        `item_id`            STRING,
        `increment_id`       STRING,
        `created_at`         TIMESTAMP,
        `price`              DOUBLE,
        `qty_ordered`        STRING,
        `total_value`        DOUBLE,
        `grand_total`        STRING,
        `discount_amount`    STRING,
        `status`             STRING,
        `payment_method`     STRING,
        `sku`                STRING,
        `Customer ID`        STRING,
        `category_name_1`    STRING
    )
    COMMENT 'E-commerce sales order details'
    PARTITIONED BY (`year` INT, `month` INT)
    STORED AS PARQUET
""")


26/04/18 10:29:10 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/fact_order_items specified for non-external table:fact_order_items


DataFrame[]

In [10]:
spark.sql("show tables").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
| eco_data|fact_order_items|      false|
| eco_data|     sales_order|      false|
+---------+----------------+-----------+



In [11]:
df = spark.read.parquet("/home/lst/my-spark/star_schema/fact_order_items")

# 开启动态分区
spark.sql("SET hive.exec.dynamic.partition.mode=nonstrict")

# 写入分区表
df.write \
    .mode("overwrite") \
    .format("parquet") \
    .partitionBy("year", "month") \
    .saveAsTable("eco_data.fact_order_items")

26/04/18 10:30:18 WARN SetCommand: 'SET hive.exec.dynamic.partition.mode=nonstrict' might not work, since Spark doesn't support changing the Hive config dynamically. Please pass the Hive-specific config by adding the prefix spark.hadoop (e.g. spark.hadoop.hive.exec.dynamic.partition.mode) when starting a Spark application. For details, see the link: https://spark.apache.org/docs/latest/configuration.html#dynamically-loading-spark-properties.


In [12]:
print("\nfact_order_items 当前行数：")
spark.sql("SELECT count(*) as row_count FROM fact_order_items").show()


fact_order_items 当前行数：


26/04/18 10:30:49 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+---------+
|row_count|
+---------+
|   584513|
+---------+



In [13]:
print("\n前5行预览：")
spark.sql("SELECT * FROM fact_order_items LIMIT 5").show(truncate=False)


前5行预览：


26/04/18 10:31:31 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+-------+------------+-------------------+-------+-----------+-----------+-----------+---------------+--------------+--------------+----------------------+-----------+---------------+----+-----+
|item_id|increment_id|created_at         |price  |qty_ordered|total_value|grand_total|discount_amount|status        |payment_method|sku                   |Customer ID|category_name_1|year|month|
+-------+------------+-------------------+-------+-----------+-----------+-----------+---------------+--------------+--------------+----------------------+-----------+---------------+----+-----+
|605463 |100374112   |2017-11-01 00:00:00|899.0  |1          |899.0      |5286       |0              |canceled      |Easypay       |KABHED59D383669C099   |65290      |Kids & Baby    |2017|11   |
|605466 |100374112   |2017-11-01 00:00:00|1185.0 |1          |1185.0     |5286       |0              |canceled      |Easypay       |BKSPRO59C1183E61D4B   |65290      |Books          |2017|11   |
|605496 |100374129   |201

In [14]:
df2 = spark.read.parquet("/home/lst/my-spark/star_schema/dim_customers")
df2.printSchema()

root
 |-- Customer ID: string (nullable = true)
 |-- Customer Since: string (nullable = true)



In [15]:
df3 = spark.read.parquet("/home/lst/my-spark/star_schema/dim_products")
df3.printSchema()

root
 |-- sku: string (nullable = true)
 |-- category_name_1: string (nullable = true)



In [16]:
df4 = spark.read.parquet("/home/lst/my-spark/star_schema/dim_status")
df4.printSchema()

root
 |-- status: string (nullable = true)



In [17]:
df5 = spark.read.parquet("/home/lst/my-spark/star_schema/dim_time")
df5.printSchema()

root
 |-- created_at: timestamp (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [19]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.dim_customers (
        `Customer ID`       STRING COMMENT '客户唯一标识',
        `Customer Since`    STRING COMMENT '客户注册日期'
    )
    COMMENT 'Customer dimension table'
    STORED AS PARQUET
""")

# 5. 创建维度表：dim_products
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.dim_products (
        `sku`               STRING COMMENT '产品SKU',
        `category_name_1`   STRING COMMENT '一级分类名称'
    )
    COMMENT 'Product dimension table'
    STORED AS PARQUET
""")

# 6. 创建维度表：dim_status
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.dim_status (
        `status`            STRING COMMENT '订单状态'
    )
    COMMENT 'Order status dimension table'
    STORED AS PARQUET
""")

# 7. 创建维度表：dim_time
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.dim_time (
        `created_at`        TIMESTAMP COMMENT '时间戳',
        `year`              INT COMMENT '年份',
        `month`             INT COMMENT '月份'
    )
    COMMENT 'Time dimension table'
    STORED AS PARQUET
""")

print("数据库 eco_data 及所有表（fact_order_items + 四张维度表）创建成功！")


数据库 eco_data 及所有表（fact_order_items + 四张维度表）创建成功！


26/04/18 10:52:33 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/dim_customers specified for non-external table:dim_customers
26/04/18 10:52:33 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/dim_products specified for non-external table:dim_products
26/04/18 10:52:33 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/dim_status specified for non-external table:dim_status
26/04/18 10:52:33 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/dim_time specified for non-external table:dim_time


In [21]:
df_customers = spark.read.parquet("/home/lst/my-spark/star_schema/dim_customers")
df_customers.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("eco_data.dim_customers")
print("dim_customers 导入完成")


df_products = spark.read.parquet("/home/lst/my-spark/star_schema/dim_products")
df_products.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("eco_data.dim_products")
print("dim_products 导入完成")


df_status = spark.read.parquet("/home/lst/my-spark/star_schema/dim_status")
df_status.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("eco_data.dim_status")
print("dim_status 导入完成")


df_time = spark.read.parquet("/home/lst/my-spark/star_schema/dim_time")
df_time.write \
    .mode("overwrite") \
    .format("parquet") \
    .saveAsTable("eco_data.dim_time")
print("dim_time 导入完成")

print("所有维度表导入成功！")

dim_customers 导入完成
dim_products 导入完成
dim_status 导入完成
dim_time 导入完成
所有维度表导入成功！


In [22]:
print("=== 当前 4 个维度表 + 事实表行数验证 ===")
spark.sql("""
SELECT 'dim_products'     as table_name, count(*) as row_count FROM dim_products
UNION ALL
SELECT 'dim_customers'    as table_name, count(*) FROM dim_customers
UNION ALL
SELECT 'dim_time'         as table_name, count(*) FROM dim_time
UNION ALL
SELECT 'dim_status'       as table_name, count(*) FROM dim_status
UNION ALL
SELECT 'fact_order_items' as table_name, count(*) FROM fact_order_items
""").show()

=== 当前 4 个维度表 + 事实表行数验证 ===


26/04/18 10:55:04 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+----------------+---------+
|      table_name|row_count|
+----------------+---------+
|    dim_products|   169770|
|   dim_customers|   230652|
|        dim_time|     1578|
|      dim_status|       34|
|fact_order_items|   584513|
+----------------+---------+



In [23]:
print("=== 月销售额趋势分析 ===")
spark.sql("""
SELECT 
    year,
    month,
    COUNT(DISTINCT increment_id) AS order_count,
    SUM(total_value) AS total_sales,
    AVG(total_value) AS avg_order_value,
    SUM(discount_amount) AS total_discount
FROM fact_order_items
GROUP BY year, month
ORDER BY year, month
""").show(20)

=== 月销售额趋势分析 ===


26/04/18 10:55:49 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+----+-----+-----------+--------------------+------------------+--------------------+
|year|month|order_count|         total_sales|   avg_order_value|      total_discount|
+----+-----+-----------+--------------------+------------------+--------------------+
|2016|    7|       7265|        3.52979853E7|3994.7923607967405|           124341.01|
|2016|    8|       9946|       4.297050035E7|3725.2275986129175|            130831.2|
|2016|    9|      13135|        8.12190968E7| 5262.690131536318|   5111354.689999997|
|2016|   10|      10880|       7.604634706E7| 5794.448876866809|           256290.54|
|2016|   11|      55450|2.4097296060999998E8| 3368.931895341684| 2.962164504000005E7|
|2016|   12|      10992| 7.448707996999998E7| 5537.249477401128|  51405.020000000004|
|2017|    1|      10073| 9.965062794000001E7| 7552.150658582797|   744978.9800000001|
|2017|    2|       8758|        6.93077414E7| 5843.330360003373|  100580.98010000002|
|2017|    3|      13542|       1.063677603E8| 5445.820

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("build_buckets") \
    .enableHiveSupport() \
    .getOrCreate()

26/04/27 16:51:49 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 16:51:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 16:51:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.fact_order_items (
        `item_id`            STRING,
        `increment_id`       STRING,
        `created_at`         TIMESTAMP,
        `price`              DOUBLE,
        `qty_ordered`        STRING,
        `total_value`        DOUBLE,
        `grand_total`        STRING,
        `discount_amount`    STRING,
        `status`             STRING,
        `payment_method`     STRING,
        `sku`                STRING,
        `Customer ID`        STRING,
        `category_name_1`    STRING
    )
    COMMENT 'E-commerce sales order details'
    PARTITIONED BY (`year` INT, `month` INT)
    CLUSTERED BY (`Customer ID`) INTO 16 BUCKETS   -- 注意列名有空格，需用反引号括起来
    STORED AS PARQUET
""")

26/04/27 17:04:09 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/27 17:04:09 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


DataFrame[]

In [12]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS eco_data.fact_order_items_bucketing (
        item_id            STRING,
        increment_id       STRING,
        created_at         TIMESTAMP,
        price              DOUBLE,
        qty_ordered        STRING,
        total_value        DOUBLE,
        grand_total        STRING,
        discount_amount    STRING,
        status             STRING,
        payment_method     STRING,
        sku                STRING,
        customer_id        STRING,
        category_name_1    STRING
    )
    COMMENT 'E-commerce sales order details with bucketing'
    PARTITIONED BY (year INT, month INT)
    CLUSTERED BY (customer_id) INTO 16 BUCKETS
    STORED AS PARQUET
""")

26/04/27 17:11:00 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/27 17:11:00 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/04/27 17:11:00 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/27 17:11:00 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/04/27 17:11:00 WARN HiveMetaStore: Location: file:/home/lst/spark-3.5.8/warehouse/eco_data.db/fact_order_items_bucketing specified for non-external table:fact_order_items_bucketing


DataFrame[]

In [16]:
spark.sql("show tables;").show(truncate=False)

+---------+--------------------------+-----------+
|namespace|tableName                 |isTemporary|
+---------+--------------------------+-----------+
|eco_data |dim_customers             |false      |
|eco_data |dim_products              |false      |
|eco_data |dim_status                |false      |
|eco_data |dim_time                  |false      |
|eco_data |fact_order_items          |false      |
|eco_data |fact_order_items_bucketing|false      |
|eco_data |sales_order               |false      |
+---------+--------------------------+-----------+



In [17]:
spark.sql("""
    INSERT OVERWRITE TABLE eco_data.fact_order_items_bucketing
    PARTITION (year, month)
    SELECT
        item_id,
        increment_id,
        created_at,
        price,
        qty_ordered,
        total_value,
        grand_total,
        discount_amount,
        status,
        payment_method,
        sku,
        `Customer ID` AS customer_id,   -- 原表列名带空格，此处重命名
        category_name_1,
        year,
        month
    FROM eco_data.fact_order_items
""")

26/04/27 17:35:41 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

DataFrame[]

In [18]:
print("带分桶表行数：")
spark.sql("SELECT count(*) as row_count FROM fact_order_items_bucketing").show()

print("\n查看分桶信息：")
spark.sql("DESCRIBE FORMATTED fact_order_items_bucketing").show(truncate=False)

带分桶表行数：


26/04/27 17:37:54 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

+---------+
|row_count|
+---------+
|   584513|
+---------+


查看分桶信息：
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|item_id                |string   |NULL   |
|increment_id           |string   |NULL   |
|created_at             |timestamp|NULL   |
|price                  |double   |NULL   |
|qty_ordered            |string   |NULL   |
|total_value            |double   |NULL   |
|grand_total            |string   |NULL   |
|discount_amount        |string   |NULL   |
|status                 |string   |NULL   |
|payment_method         |string   |NULL   |
|sku                    |string   |NULL   |
|customer_id            |string   |NULL   |
|category_name_1        |string   |NULL   |
|year                   |int      |NULL   |
|month                  |int      |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|year                   |int      |NULL   |
|month

In [19]:
print("=== 分桶表结构和分桶信息 ===")
spark.sql("DESCRIBE FORMATTED fact_order_items_bucketing").show(truncate=False)

=== 分桶表结构和分桶信息 ===
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|item_id                |string   |NULL   |
|increment_id           |string   |NULL   |
|created_at             |timestamp|NULL   |
|price                  |double   |NULL   |
|qty_ordered            |string   |NULL   |
|total_value            |double   |NULL   |
|grand_total            |string   |NULL   |
|discount_amount        |string   |NULL   |
|status                 |string   |NULL   |
|payment_method         |string   |NULL   |
|sku                    |string   |NULL   |
|customer_id            |string   |NULL   |
|category_name_1        |string   |NULL   |
|year                   |int      |NULL   |
|month                  |int      |NULL   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|year                   |int      |NULL   |
|month                  |int      |NULL   |
|            

In [23]:
spark.sql("show tables").show(truncate=False)
spark.sql("select * from fact_order_items_bucketing").show(truncate=False)

+---------+--------------------------+-----------+
|namespace|tableName                 |isTemporary|
+---------+--------------------------+-----------+
|eco_data |dim_customers             |false      |
|eco_data |dim_products              |false      |
|eco_data |dim_status                |false      |
|eco_data |dim_time                  |false      |
|eco_data |fact_order_items          |false      |
|eco_data |fact_order_items_bucketing|false      |
|eco_data |sales_order               |false      |
+---------+--------------------------+-----------+

+-------+------------+-------------------+-------+-----------+-----------+-----------+---------------+--------------+---------------+-------------------+-----------+-----------------+----+-----+
|item_id|increment_id|created_at         |price  |qty_ordered|total_value|grand_total|discount_amount|status        |payment_method |sku                |customer_id|category_name_1  |year|month|
+-------+------------+-------------------+------

26/04/27 17:42:30 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.loopJoinOrderedResult(MetaStoreDirectSql.java:955) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.getPartitionsFromPartitionIds(MetaStoreDirectSql.java:629) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.access$800(MetaStoreDirectSql.java:92) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$2.run(MetaStoreDirectSql.java:492) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql.runBatched(MetaStoreDir

In [24]:
print("=== RFM 用户分层分析 ===")
spark.sql("""
WITH rfm_base AS (
    SELECT 
        customer_id,
        MAX(created_at) AS last_purchase_date,
        COUNT(DISTINCT increment_id) AS frequency,
        SUM(total_value) AS monetary
    FROM fact_order_items_bucketing
    GROUP BY customer_id
),
rfm_scores AS (
    SELECT 
        customer_id,
        last_purchase_date,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY last_purchase_date ASC) AS R_score,   -- Recency: 最近购买越近分数越高，这里反过来排
        NTILE(5) OVER (ORDER BY frequency DESC) AS F_score,
        NTILE(5) OVER (ORDER BY monetary DESC) AS M_score
    FROM rfm_base
)
SELECT 
    customer_id,
    R_score,
    F_score,
    M_score,
    (R_score + F_score + M_score) AS RFM_score,
    CASE 
        WHEN (R_score + F_score + M_score) >= 12 THEN '高价值客户'
        WHEN (R_score + F_score + M_score) >= 9 THEN '中高价值客户'
        WHEN (R_score + F_score + M_score) >= 6 THEN '中价值客户'
        ELSE '低价值客户'
    END AS customer_segment
FROM rfm_scores
ORDER BY RFM_score DESC
LIMIT 20
""").show(truncate=False)

=== RFM 用户分层分析 ===


26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:44:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 1

+-----------+-------+-------+-------+---------+----------------+
|customer_id|R_score|F_score|M_score|RFM_score|customer_segment|
+-----------+-------+-------+-------+---------+----------------+
|99507      |5      |5      |5      |15       |高价值客户      |
|103984     |5      |5      |5      |15       |高价值客户      |
|102459     |5      |5      |5      |15       |高价值客户      |
|103471     |5      |5      |5      |15       |高价值客户      |
|103439     |5      |5      |5      |15       |高价值客户      |
|103591     |5      |5      |5      |15       |高价值客户      |
|103548     |5      |5      |5      |15       |高价值客户      |
|103632     |5      |5      |5      |15       |高价值客户      |
|103639     |5      |5      |5      |15       |高价值客户      |
|103656     |5      |5      |5      |15       |高价值客户      |
|103742     |5      |5      |5      |15       |高价值客户      |
|103725     |5      |5      |5      |15       |高价值客户      |
|103796     |5      |5      |5      |15       |高价值客户      |
|103776     |5      |5   

In [25]:
print("=== 数据倾斜检测 - 各字段 Top 10 分布 ===")

# 1. customer_id 分布（最容易倾斜）
spark.sql("""
SELECT 
    `Customer ID` as customer_id,
    COUNT(*) as record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM fact_order_items
GROUP BY `Customer ID`
ORDER BY record_count DESC
LIMIT 10
""").show(truncate=False)

# 2. sku 分布
spark.sql("""
SELECT 
    sku,
    COUNT(*) as record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM fact_order_items
GROUP BY sku
ORDER BY record_count DESC
LIMIT 10
""").show(truncate=False)

# 3. category_name_1 分布
spark.sql("""
SELECT 
    category_name_1,
    COUNT(*) as record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage
FROM fact_order_items
GROUP BY category_name_1
ORDER BY record_count DESC
LIMIT 10
""").show(truncate=False)

=== 数据倾斜检测 - 各字段 Top 10 分布 ===


26/04/27 17:51:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:33 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectS

+-----------+------------+----------+
|customer_id|record_count|percentage|
+-----------+------------+----------+
|85775      |2524        |0.43      |
|163        |2349        |0.40      |
|35         |1877        |0.32      |
|33         |1397        |0.24      |
|31025      |1369        |0.23      |
|806        |1310        |0.22      |
|1404       |1269        |0.22      |
|767        |1234        |0.21      |
|820        |1190        |0.20      |
|58         |1182        |0.20      |
+-----------+------------+----------+



26/04/27 17:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:33 WARN ObjectStore: Falling back to ORM path due to direct SQL failure (this is not an error): class org.apache.derby.impl.jdbc.EmbedClob cannot be cast to class java.lang.String (org.apache.derby.impl.jdbc.EmbedClob is in unnamed module of loader 'app'; java.lang.String is in module java.base of loader 'bootstrap') at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectSql.java:632) at org.apache.hadoop.hive.metastore.MetaStoreDirectSql$3.apply(MetaStoreDirectS

+-----------------------------+------------+----------+
|sku                          |record_count|percentage|
+-----------------------------+------------+----------+
|MATSAM59DB75ADB2F80          |3775        |0.65      |
|Al Muhafiz Sohan Halwa Almond|2258        |0.39      |
|emart_00-7                   |2027        |0.35      |
|kcc_krone deal               |1894        |0.32      |
|infinix_Zero 4-Grey          |1793        |0.31      |
|emart_00-1                   |1391        |0.24      |
|MATSAM59DB757FB47A2          |1273        |0.22      |
|Rubian_U8 Smart Watch        |1233        |0.21      |
|unilever_Deal-6              |1213        |0.21      |
|APPNAT5A0A01860CE92          |1173        |0.20      |
+-----------------------------+------------+----------+

+-----------------+------------+----------+
|category_name_1  |record_count|percentage|
+-----------------+------------+----------+
|Mobiles & Tablets|115710      |19.80     |
|Men's Fashion    |92220       |15.78  

26/04/27 17:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/27 17:51:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
